In [1]:
import numpy as np

def train_test_split(X,y,test_size=0.2,random_state=None):

    if random_state is not None:
        np.random.seed(random_state)
    n_samples=X.shape[0]
    indices=np.arange(n_samples)
    np.random.shuffle(indices)

    test_size=int(n_samples*test_size)
    test_indices=indices[:test_size]
    train_indices=indices[test_size:]

    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]



def standard_scaler(X_train,X_test):

    mean=np.mean(X_train,axis=0)
    std=np.std(X_train,axis=0)
    X_train_scaled=(X_train-mean)/std
    X_test_scaled=(X_test-mean)/std

    return X_train_scaled,X_test_scaled



def mean_squared_error(y_true,y_pred):
    return np.mean((y_true-y_pred)**2)



def r2_score(y_true,y_pred):
    ss_total=np.sum((y_true-np.mean(y_true))**2)
    ss_residual=np.sum((y_true-y_pred)**2)

    return 1-(ss_residual/ss_total)



class MulLinRegression:

    def __init__(self,method='analytic',learning_rate=0.01,n_iterations=10000):

        self.method=method
        self.learning_rate=learning_rate
        self.n_iterations=n_iterations

        self.coef_=None
        self.intercept_=None
        self._theta=None


    def fit(self,X,y):

        if not isinstance(X,np.ndarray) or X.ndim != 2:
            raise ValueError('X must be a 2-d numpy array')
        if not isinstance(y,np.ndarray) or y.ndim != 1:
            raise ValueError('y must be a 2-d numpy array')
        if X.shape[0] != y.shape[0]:
            raise ValueError('X and y must have the same number of rows')


        X_b=np.c_[np.ones((X.shape[0],1)),X]


        if self.method=='analytic':
            X_b_T=X_b.T
            self._theta=np.linalg.inv(X_b_T@X_b+1e-6*np.eye(X_b.shape[1]))@X_b_T@y


        elif self.method=='gradient_descent':
            n_samples=X.shape[0]
            self._theta=np.random.randn(X_b.shape[1])

            for _ in range(self.n_iterations):
                gradients=2/n_samples*X_b.T@(X_b@self._theta-y)
                self._theta-=self.learning_rate*gradients


        else:
            raise ValueError('Unknown method')


        self.intercept_=self._theta[0]
        self.coef_=self._theta[1:]
        return self



    def predict(self,X):

        if self._theta is None:
            raise RuntimeError('No theta has been trained yet.')

        if not isinstance(X,np.ndarray) or X.ndim != 2:
            raise ValueError('X must be a 2-d numpy array')

        if X.shape[1] != len(self.coef_):
            raise ValueError('X must have the same number of columns as the number of features')


        X_b=np.c_[np.ones((X.shape[0],1)),X]

        return X_b@self._theta


def load_boston_housing():

    import pandas as pd
    import urllib.request


    data_url = "http://lib.stat.cmu.edu/datasets/boston"

    raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
    target = raw_df.values[1::2, 2]

    return data, target


X, y = load_boston_housing()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled, X_test_scaled = standard_scaler(X_train, X_test)


model_analytic = MulLinRegression(method='analytic')

model_analytic.fit(X_train_scaled, y_train)

y_train_pred_analytic = model_analytic.predict(X_train_scaled)
y_test_pred_analytic = model_analytic.predict(X_test_scaled)

train_mse_analytic = mean_squared_error(y_train, y_train_pred_analytic)
test_mse_analytic = mean_squared_error(y_test, y_test_pred_analytic)
train_r2_analytic = r2_score(y_train, y_train_pred_analytic)
test_r2_analytic = r2_score(y_test, y_test_pred_analytic)

print("解析解训练集 MSE：", round(train_mse_analytic, 2))
print("解析解测试集 MSE：", round(test_mse_analytic, 2))
print("解析解训练集r2:", round(train_r2_analytic, 2))
print("解析解测试集r2:", round(test_r2_analytic, 2))


model_gd = MulLinRegression(
    method='gradient_descent',
    learning_rate=0.01,
    n_iterations=10000
)

model_gd.fit(X_train_scaled, y_train)

y_train_pred_gd = model_gd.predict(X_train_scaled)
y_test_pred_gd = model_gd.predict(X_test_scaled)


train_mse_gd = mean_squared_error(y_train, y_train_pred_gd)
test_mse_gd = mean_squared_error(y_test, y_test_pred_gd)
train_r2_gd = r2_score(y_train, y_train_pred_gd)
test_r2_gd = r2_score(y_test, y_test_pred_gd)


print("梯度下降训练集 MSE:", round(train_mse_gd, 2))
print("梯度下降测试集 MSE:", round(test_mse_gd, 2))
print("梯度下降训练集r2:", round(train_r2_gd, 2))
print("梯度下降测试集r2:", round(test_r2_gd, 2))


解析解训练集 MSE： 21.61
解析解测试集 MSE： 24.4
解析解训练集r2: 0.75
解析解测试集r2: 0.67
梯度下降训练集 MSE: 21.61
梯度下降测试集 MSE: 24.4
梯度下降训练集r2: 0.75
梯度下降测试集r2: 0.67
